# Qwen Reconciliation Resume: q1p5 Long Only

Use this notebook after the original reconciliation run already completed q3b long training and q1p5 grid search, but q1p5 long failed because Google Drive filled up or Drive FUSE errored.

This notebook intentionally skips q3b and all grid training. It stages the q1p5 corpus/frozen file to local `/content`, trains the q1p5 long winner locally, then copies only essential final artifacts back to Drive.


In [ ]:
# Cell 1 - Setup, repo, paths
from google.colab import drive
drive.mount('/content/drive')

import glob, json, os, shutil, subprocess, sys, time
from pathlib import Path


def run(cmd, **kwargs):
    cmd = [str(x) for x in cmd]
    if cmd and cmd[0] == sys.executable and (len(cmd) == 1 or cmd[1] != '-u'):
        cmd.insert(1, '-u')
    env = os.environ.copy()
    env.update(kwargs.pop('env', {}) or {})
    env['PYTHONUNBUFFERED'] = '1'
    print('$ ' + ' '.join(cmd), flush=True)
    return subprocess.run(cmd, check=True, env=env, **kwargs)

run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'accelerate>=1.0', 'datasets>=3.0', 'pyarrow>=17', 'tqdm>=4.66',
    'zstandard', 'bitsandbytes>=0.43', 'gguf', 'safetensors>=0.4',
    'hf_transfer>=0.1.9',
])

os.environ.setdefault('HF_HUB_ENABLE_HF_TRANSFER', '1')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('PYTHONUNBUFFERED', '1')

try:
    from google.colab import userdata
    _hf_token = userdata.get('HF_TOKEN')
except Exception:
    _hf_token = None
if _hf_token:
    os.environ['HF_TOKEN'] = _hf_token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = _hf_token
    print('HF_TOKEN loaded from Colab secrets.')
else:
    print('No HF_TOKEN secret found; public Hub downloads still work.')

import torch
assert torch.cuda.is_available(), 'No CUDA: set Runtime -> Change runtime type -> GPU'
GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {GPU_NAME}  VRAM: {VRAM_GB:.1f} GB')

REPO = 'https://github.com/joshuahickscorp/dismantle.git'
BRANCH = 'main'
if not os.path.exists('/content/dismantle'):
    run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO, '/content/dismantle'])
else:
    run(['git', '-C', '/content/dismantle', 'fetch', 'origin', BRANCH, '--depth', '1'])
    run(['git', '-C', '/content/dismantle', 'checkout', BRANCH])
    run(['git', '-C', '/content/dismantle', 'reset', '--hard', f'origin/{BRANCH}'])
os.chdir('/content/dismantle')
print('repo:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())

DRIVE_ROOT = Path('/content/drive/MyDrive/dismantle')
ARTIFACT_DIR = DRIVE_ROOT / 'qwen_reconciliation'
DRIVE_CKPT_ROOT = ARTIFACT_DIR / 'checkpoints'
LOCAL_ROOT = Path('/content/q1p5_resume_local')
LOCAL_CORPUS = LOCAL_ROOT / 'qwen1p5_corpus'
LOCAL_FROZEN = LOCAL_ROOT / 'q1p5_frozen.npz'
LOCAL_CKPT = LOCAL_ROOT / 'q1p5_e6_b2_wide_b2_h16_ff60_lr5e-4_rd020_cw20_long'

DRIVE_Q1_CORPUS = DRIVE_ROOT / 'qwen1p5_corpus'
DRIVE_Q1_FROZEN = ARTIFACT_DIR / 'qwen1p5_frozen.npz'
DRIVE_Q1_LONG = DRIVE_CKPT_ROOT / 'q1p5_e6_b2_wide_b2_h16_ff60_lr5e-4_rd020_cw20_long'
DRIVE_Q1_HEAD = DRIVE_Q1_LONG / 'head_final.safetensors'

DRIVE_Q3_HEAD = DRIVE_CKPT_ROOT / 'q3b_e6_b1_wide_b1_h16_ff60_lr5e-4_rd020_cw20_long' / 'head_final.safetensors'

for d in (LOCAL_ROOT, LOCAL_CORPUS, LOCAL_CKPT, DRIVE_Q1_LONG):
    d.mkdir(parents=True, exist_ok=True)

q1_shards = sorted(DRIVE_Q1_CORPUS.glob('shard_*.parquet'))
assert q1_shards, f'missing q1p5 corpus shards: {DRIVE_Q1_CORPUS}'
assert DRIVE_Q1_FROZEN.exists(), f'missing q1p5 frozen baseline: {DRIVE_Q1_FROZEN}'
print(f'q1p5 Drive shards: {len(q1_shards)}')
print(f'q1p5 frozen: {DRIVE_Q1_FROZEN} ({DRIVE_Q1_FROZEN.stat().st_size/1e9:.2f} GB)')
if DRIVE_Q3_HEAD.exists():
    print(f'q3b final head exists: {DRIVE_Q3_HEAD} ({DRIVE_Q3_HEAD.stat().st_size/1e9:.2f} GB)')
else:
    print(f'WARNING: q3b final head not found at {DRIVE_Q3_HEAD}')

local_free = shutil.disk_usage('/content').free / 1e9
print(f'/content free: {local_free:.1f} GB')


In [ ]:
# Cell 2 - Delete non-essential Drive files now (no dry run)
from pathlib import Path

KEEP = {
    DRIVE_Q3_HEAD.resolve(),
    DRIVE_Q1_FROZEN.resolve(),
}

delete_paths = []

# q3b corpus and frozen are expendable once q3b long head exists.
delete_paths += list((DRIVE_ROOT / 'qwen3b_corpus').glob('shard_*.parquet'))
delete_paths.append(ARTIFACT_DIR / 'qwen3b_frozen.npz')

# Historical/rolling checkpoints are not used for resume.
delete_paths += list(DRIVE_CKPT_ROOT.glob('**/step_*.npz'))
delete_paths += list(DRIVE_CKPT_ROOT.glob('**/latest.npz'))

# Grid head files are selection history only. Keep tau.json for record.
delete_paths += list(DRIVE_CKPT_ROOT.glob('*_grid/head_final.safetensors'))

# Remove any failed partial q1p5 long artifacts before the local rerun.
delete_paths += list(DRIVE_Q1_LONG.glob('*.npz'))
if DRIVE_Q1_HEAD.exists():
    delete_paths.append(DRIVE_Q1_HEAD)

seen = set()
total = 0
count = 0
errors = []
for p in delete_paths:
    try:
        p = p.resolve()
        if p in seen or p in KEEP or not p.exists() or p.is_dir():
            continue
        seen.add(p)
        size = p.stat().st_size
        p.unlink()
        total += size
        count += 1
        print('DELETED', f'{size/1e9:7.2f} GB', p)
    except Exception as e:
        errors.append((str(p), repr(e)))
        print('ERROR ', p, repr(e))

for d in sorted(DRIVE_CKPT_ROOT.glob('**/*'), key=lambda x: len(str(x)), reverse=True):
    if d.is_dir():
        try:
            d.rmdir()
        except OSError:
            pass

print(f'\ndeleted files: {count}')
print(f'freed from files: {total/1e9:.2f} GB')
print(f'errors: {len(errors)}')
if errors[:10]:
    print(errors[:10])
print('If Drive quota still shows full, open Drive web UI and empty Trash before training.')


In [ ]:
# Cell 3 - Stage q1p5 inputs to local /content
import time


def copy2_retry(src: Path, dst: Path, tries=4, sleep_s=5.0):
    dst.parent.mkdir(parents=True, exist_ok=True)
    src_size = src.stat().st_size
    if dst.exists() and dst.stat().st_size == src_size:
        return False
    last = None
    tmp = dst.with_suffix(dst.suffix + '.tmp')
    for attempt in range(1, tries + 1):
        try:
            if tmp.exists():
                tmp.unlink()
            shutil.copy2(src, tmp)
            if tmp.stat().st_size != src_size:
                raise IOError(f'size mismatch: {tmp.stat().st_size} != {src_size}')
            os.replace(tmp, dst)
            return True
        except Exception as e:
            last = e
            print(f'copy retry {attempt}/{tries} failed for {src.name}: {e}', flush=True)
            time.sleep(sleep_s * attempt)
    raise RuntimeError(f'failed to copy {src} -> {dst}: {last}')

LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_CORPUS.mkdir(parents=True, exist_ok=True)

print('copy frozen -> local')
changed = copy2_retry(DRIVE_Q1_FROZEN, LOCAL_FROZEN)
print(('copied' if changed else 'already local'), LOCAL_FROZEN, f'{LOCAL_FROZEN.stat().st_size/1e9:.2f} GB')

shards = sorted(DRIVE_Q1_CORPUS.glob('shard_*.parquet'))
print(f'copying/resuming {len(shards)} q1p5 shards to {LOCAL_CORPUS}')
bytes_copied = 0
for i, src in enumerate(shards, 1):
    dst = LOCAL_CORPUS / src.name
    before = dst.stat().st_size if dst.exists() else 0
    changed = copy2_retry(src, dst)
    if changed:
        bytes_copied += src.stat().st_size
    if i == 1 or i % 25 == 0 or i == len(shards):
        print(f'  staged {i}/{len(shards)} shards; copied this run {bytes_copied/1e9:.2f} GB', flush=True)

local_shards = sorted(LOCAL_CORPUS.glob('shard_*.parquet'))
assert len(local_shards) == len(shards), f'local shard count mismatch: {len(local_shards)} != {len(shards)}'
print(f'local q1p5 corpus ready: {len(local_shards)} shards')
print(f'/content free after staging: {shutil.disk_usage("/content").free/1e9:.1f} GB')


In [ ]:
# Cell 4 - Train q1p5 long winner locally, then copy only final essentials to Drive
import shutil

LOCAL_CKPT.mkdir(parents=True, exist_ok=True)
local_head = LOCAL_CKPT / 'head_final.safetensors'

# Remove stale local partials, but keep a completed local head if present.
if not local_head.exists():
    for pat in ('step_*.npz', 'latest.npz'):
        for p in LOCAL_CKPT.glob(pat):
            p.unlink()

if local_head.exists():
    print(f'local q1p5 long head already exists: {local_head} ({local_head.stat().st_size/1e9:.2f} GB)')
else:
    cmd = [
        sys.executable, 'colab/eagle5_train_pytorch.py',
        '--corpus-dir', str(LOCAL_CORPUS),
        '--frozen', str(LOCAL_FROZEN),
        '--ckpt-dir', str(LOCAL_CKPT),
        '--epochs', '20',
        '--batch-size', '24',
        '--seq-len', '16',
        '--lr', '0.0005',
        '--capture-layer', '22',
        '--max-rows', '8000',
        '--max-row-tokens', '192',
        '--sparsity-head', 'off',
        '--seed', '0',
        '--calib-loss-weight', '0.2',
        '--residual-delta-loss-weight', '0.02',
        '--num-blocks', '2',
        '--head-heads', '16',
        '--head-ff-mult', '6.0',
        '--save-safetensors',
    ]
    run(cmd)
    assert local_head.exists(), f'trainer did not write {local_head}'

DRIVE_Q1_LONG.mkdir(parents=True, exist_ok=True)
print('copy final q1p5 head -> Drive')
copy2_retry(local_head, DRIVE_Q1_HEAD)
print(f'Drive q1p5 head: {DRIVE_Q1_HEAD} ({DRIVE_Q1_HEAD.stat().st_size/1e9:.2f} GB)')

local_log = LOCAL_CKPT / 'log.jsonl'
if local_log.exists():
    copy2_retry(local_log, DRIVE_Q1_LONG / 'log.jsonl')
    print('copied log.jsonl')

# Do not copy latest.npz. The final safetensors is the deployable artifact.


In [ ]:
# Cell 5 - Evaluate q1p5 long locally and copy result JSONs to Drive
LOCAL_TAU = LOCAL_CKPT / 'tau.json'
LOCAL_FRONTIER = LOCAL_CKPT / 'frontier.json'

run([
    sys.executable, 'colab/eagle5_tau_eval_pytorch.py',
    '--ckpt', str(local_head),
    '--frozen', str(LOCAL_FROZEN),
    '--corpus', str(LOCAL_CORPUS),
    '--out', str(LOCAL_TAU),
    '--depth', '4',
    '--max-windows', '6000',
    '--max-row-tokens', '192',
    '--base-tps', '55.0',
    '--w4a8-multiplier', '1.15',
    '--spec-efficiency', '0.7',
])
copy2_retry(LOCAL_TAU, DRIVE_Q1_LONG / 'tau.json')

run([
    sys.executable, 'colab/eagle5_frontier_policy.py',
    '--ckpt', str(local_head),
    '--frozen', str(LOCAL_FROZEN),
    '--corpus', str(LOCAL_CORPUS),
    '--out', str(LOCAL_FRONTIER),
    '--max-depth', '12',
    '--depths', '4,6,8,12',
    '--lattice-widths', '2,3,4',
    '--max-windows', '6000',
    '--max-row-tokens', '192',
    '--base-tps', '55.0',
    '--w4a8-multiplier', '1.15',
    '--spec-efficiency', '0.7',
])
copy2_retry(LOCAL_FRONTIER, DRIVE_Q1_LONG / 'frontier.json')

with open(LOCAL_FRONTIER) as f:
    frontier = json.load(f)
best = frontier['policies']['best_deployable']
summary = {
    'q3b_head': str(DRIVE_Q3_HEAD) if DRIVE_Q3_HEAD.exists() else None,
    'q1p5_head': str(DRIVE_Q1_HEAD),
    'q1p5_best_deployable': best,
}
summary_path = ARTIFACT_DIR / 'q1p5_resume_summary.json'
summary_path.write_text(json.dumps(summary, indent=2, sort_keys=True))
print(json.dumps(summary, indent=2, sort_keys=True))
print(f'wrote {summary_path}')


In [ ]:
# Cell 6 - Optional post-success Drive cleanup
# Run this only after Cell 4 has copied q1p5 head_final.safetensors to Drive
# and Cell 5 has copied tau/frontier JSONs.
DELETE_Q1P5_CORPUS_AFTER_SUCCESS = False

if DELETE_Q1P5_CORPUS_AFTER_SUCCESS:
    assert DRIVE_Q1_HEAD.exists(), 'q1p5 final head is not on Drive yet'
    removed = 0
    freed = 0
    for p in DRIVE_Q1_CORPUS.glob('shard_*.parquet'):
        size = p.stat().st_size
        p.unlink()
        removed += 1
        freed += size
    if DRIVE_Q1_FROZEN.exists():
        size = DRIVE_Q1_FROZEN.stat().st_size
        DRIVE_Q1_FROZEN.unlink()
        removed += 1
        freed += size
    print(f'deleted q1p5 corpus/frozen: {removed} files, {freed/1e9:.2f} GB')
else:
    print('Post-success q1p5 corpus cleanup is disabled. Flip DELETE_Q1P5_CORPUS_AFTER_SUCCESS=True after final artifacts are copied.')
